# Activation Harvesting for TopK SAE — UmedPT Swin Transformer

This notebook harvests intermediate activations from the UmedPT Swin Transformer
for use in training a **TopK Sparse Autoencoder** (following Anthropic's TopK SAE recipe).

## Pipeline
1. Load 50,000 synthetic shape images (`mmm.ShapeDataset`, fallback to built-in)
2. Load the encoder directly from `encoder.pt`
3. **`register_forward_hook`** on the `PatchMergingV2` layer at the **Stage 2→3 boundary**
4. Pass all batches; hook captures `(B, H', W', C')` NHWC tensors
5. Reshape: `(B, C, H, W)` → `(B*H*W, C)`
6. Normalize: `x_norm = x / (√mean(x²) + ε) × √d_model`
7. Print `N_effective`, save to `data/dummy_activations.safetensors`

---
**PatchMerging index guide**
| `PM_INDEX` | Boundary | d_model | Spatial |
|---|---|---|---|
| 0 | Stage 1→2 | 256 | 28×28 |
| **1** | **Stage 2→3** | **512** | **14×14** |
| 2 | Stage 3→4 | 1024 | 7×7 |

## 1 — Configuration
Edit these variables to control the run.

In [ ]:
from pathlib import Path

# ── Edit these ────────────────────────────────────────────────────────────────
CHECKPOINT   = "../encoder-1.0.4.pt/encoder.pt"   # relative to this notebook
N_SAMPLES    = 50_000
BATCH_SIZE   = 32
IMAGE_SIZE   = 224
PM_INDEX     = 1       # 0=Stage1→2  1=Stage2→3 (default)  2=Stage3→4
OUTPUT_PATH  = "../data/dummy_activations.safetensors"
DEVICE       = "cuda:0"   # or "cpu"
# ─────────────────────────────────────────────────────────────────────────────

EPS = 1e-8
print(f"Checkpoint : {Path(CHECKPOINT).resolve()}")
print(f"Output     : {Path(OUTPUT_PATH).resolve()}")

: 

## 2 — Imports & sys.path

In [ ]:
import os
import sys
import math
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from safetensors.torch import save_file

# ── Make mmm / m3_sdk importable ──────────────────────────────────────────────
# Adjust these if UmedPT and UmedPT-sdk live elsewhere
_repo_root = Path("../").resolve()   # assuming notebook is at phase-1-topk/
for _pkg in ("UmedPT", "UmedPT-sdk"):
    _p = str(_repo_root / _pkg)
    if Path(_p).exists() and _p not in sys.path:
        sys.path.insert(0, _p)
        print(f"Added to sys.path: {_p}")

os.environ.setdefault("MMM_LICENSE_ACCEPTED", "i accept")

print(f"\nPyTorch {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
print(f"Using device   : {device}")

## 3 — Dataset

Try `mmm.data_loading.synthetic.shape_dataset.ShapeDataset` first (50 k samples),
falling back to a built-in procedural generator.

In [ ]:
_IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


class _MMMShapeWrapper(Dataset):
    """Extracts the 'image' tensor from mmm ShapeDataset and applies ImageNet norm."""
    def __init__(self, base):
        self.base = base
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        s   = self.base[idx]
        img = s["image"] if isinstance(s, dict) else s[0]
        return (img.float() - _IMAGENET_MEAN) / _IMAGENET_STD, 0


class SyntheticShapeDataset(Dataset):
    """Deterministic fallback: circles / squares / triangles / crosses."""
    SHAPES = ["circle", "square", "triangle", "cross"]

    def __init__(self, n_samples=50_000, image_size=224):
        self.n  = n_samples
        self.sz = image_size

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        H = W = self.sz
        shape = self.SHAPES[idx % 4]
        g     = torch.Generator().manual_seed(idx)
        rnd   = lambda: torch.rand(1, generator=g).item()

        img = torch.zeros(3, H, W)
        img[0].fill_(rnd() * 0.3 + 0.6)
        img[1].fill_(rnd() * 0.3 + 0.5)
        img[2].fill_(rnd() * 0.3 + 0.6)

        cx, cy = int(rnd()*0.4*W + 0.3*W), int(rnd()*0.4*H + 0.3*H)
        r      = int(rnd() * 0.15 * H + 0.1 * H)
        fg     = (rnd()*0.4, rnd()*0.3, rnd()*0.5)

        yy, xx = torch.meshgrid(torch.arange(H), torch.arange(W), indexing="ij")
        if shape == "circle":
            mask = ((xx-cx)**2 + (yy-cy)**2) <= r**2
        elif shape == "square":
            mask = ((xx-cx).abs() <= r) & ((yy-cy).abs() <= r)
        elif shape == "triangle":
            half = ((yy-cy).float() / max(r,1) * r).long()
            mask = ((yy>=cy) & (yy<=cy+r)) & ((xx-cx).abs() <= half)
        else:
            arm  = max(r//3, 2)
            mask = (((xx-cx).abs()<=arm) & ((yy-cy).abs()<=r)) | \
                   (((yy-cy).abs()<=arm) & ((xx-cx).abs()<=r))

        for c, v in enumerate(fg):
            img[c, mask] = v

        return ((img - _IMAGENET_MEAN) / _IMAGENET_STD).float(), idx % 4


# ── Load ─────────────────────────────────────────────────────────────────────
dataset = None
try:
    from mmm.data_loading.synthetic.shape_dataset import ShapeDataset
    dataset = _MMMShapeWrapper(ShapeDataset(n_samples=N_SAMPLES, image_size=IMAGE_SIZE))
    print(f"[Dataset] mmm ShapeDataset — {N_SAMPLES:,} samples")
except Exception as e:
    print(f"[Dataset] mmm ShapeDataset unavailable: {e}")

if dataset is None:
    print("[Dataset] Using built-in SyntheticShapeDataset")
    dataset = SyntheticShapeDataset(n_samples=N_SAMPLES, image_size=IMAGE_SIZE)

print(f"Dataset length : {len(dataset):,}")

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)
print(f"Batches        : {len(dataloader)}")

## 4 — Load Encoder

Load the `PyramidEncoder` module directly from `encoder.pt`.
The file is a fully serialised `torch.nn.Module` — `mmm` must be importable first.

In [ ]:
ckpt_path = Path(CHECKPOINT).resolve()
assert ckpt_path.exists(), (
    f"encoder.pt not found at {ckpt_path}\n"
    f"Adjust CHECKPOINT in cell 1."
)

encoder = torch.load(str(ckpt_path), map_location=device, weights_only=False)
encoder.eval()

print(f"Encoder type   : {type(encoder).__name__}")
print(f"Backbone type  : {type(encoder.model).__name__}")
print(f"SwinV2 type    : {type(encoder.model.wrapped_model).__name__}")

# Quick forward-pass to verify the model works
with torch.inference_mode():
    _probe = encoder(torch.rand(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device))
print(f"\nFeature pyramid levels: {len(_probe)}")
for i, f in enumerate(_probe):
    print(f"  Level {i}: {tuple(f.shape)}")

## 5 — Register Forward Hook

We hook the **`PatchMergingV2`** layer between Stage 2 and Stage 3
(`encoder.model.wrapped_model.features[4]`).

This captures activations **immediately after** the patch-merging downsampling,
before the Stage 3 transformer blocks run — a clean architectural boundary.

In [ ]:
from torchvision.models.swin_transformer import PatchMergingV2

# Navigate to the torchvision Swin backbone
swin_features = encoder.model.wrapped_model.features

# Find all PatchMergingV2 layers in order
pm_layers = [
    (i, m) for i, m in enumerate(swin_features)
    if isinstance(m, PatchMergingV2)
]

_boundary_names = {0: "Stage 1→2", 1: "Stage 2→3", 2: "Stage 3→4"}
print("PatchMerging layers found:")
for n, (fi, _) in enumerate(pm_layers):
    marker = " ← target" if n == PM_INDEX else ""
    print(f"  PM-{n}  {_boundary_names.get(n, '')}  features[{fi}]{marker}")

# Probe the target layer to see its output shape
captured = {}

def _hook_fn(module, input, output):
    """
    Swin internals use NHWC layout.
    output: (B, H', W', C')  — we store raw; reshape happens in the loop.
    """
    captured["feat"] = output.detach().cpu()

target_pm_module = pm_layers[PM_INDEX][1]
hook_handle      = target_pm_module.register_forward_hook(_hook_fn)

# Probe output shape
with torch.inference_mode():
    encoder(torch.rand(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device))

probe_feat                    = captured["feat"]   # (1, H', W', C')
_, H_FEAT, W_FEAT, D_MODEL    = probe_feat.shape
N_SPATIAL                     = H_FEAT * W_FEAT

print(f"\nHook output shape : (B, {H_FEAT}, {W_FEAT}, {D_MODEL})  [NHWC]")
print(f"d_model           : {D_MODEL}")
print(f"Tokens per image  : {N_SPATIAL}  ({H_FEAT}×{W_FEAT})")

## 6 — N_effective

$$N_{\text{effective}} = N_{\text{images}} \times H' \times W'$$

In [ ]:
N_images    = len(dataset)
N_effective = N_images * N_SPATIAL
est_gb      = N_effective * D_MODEL * 4 / 1e9   # float32

print(f"N_images    = {N_images:,}")
print(f"Tokens/img  = {N_SPATIAL}  ({H_FEAT}×{W_FEAT})")
print(f"d_model     = {D_MODEL}")
print()
print(f"N_effective = {N_effective:,}")
print(f"Est. size   = {est_gb:.1f} GB  (float32)")

## 7 — Harvest & Flatten

For each batch:
1. Run forward pass — hook fires, stores `(B, H', W', C')` NHWC
2. Permute to `(B, C, H, W)` (NCHW)
3. Flatten to `(B*H*W, C)`

In [ ]:
all_acts = []
t0       = time.time()

encoder.eval()
with torch.inference_mode():
    for batch_idx, (images, _labels) in enumerate(dataloader):
        # Forward pass — hook fires and populates captured["feat"]
        encoder(images.to(device))

        feat = captured["feat"]              # (B, H', W', C')  NHWC
        B, H, W, C = feat.shape

        # ── Reshape: (B, C, H, W) → (B*H*W, C) ──────────────────────────
        feat_nchw = feat.permute(0, 3, 1, 2)               # (B, C, H, W)
        flat      = feat_nchw.permute(0, 2, 3, 1).reshape(B * H * W, C)  # (B*H*W, C)

        all_acts.append(flat)

        if (batch_idx + 1) % 100 == 0 or (batch_idx + 1) == len(dataloader):
            n_so_far = sum(a.shape[0] for a in all_acts)
            elapsed  = time.time() - t0
            print(f"  batch {batch_idx + 1:>5} / {len(dataloader)}  |  "
                  f"{n_so_far:>12,} vectors  |  {elapsed:.1f}s")

hook_handle.remove()   # always clean up

activations = torch.cat(all_acts, dim=0).float()
del all_acts

print(f"\nRaw activations shape : {tuple(activations.shape)}")
print(f"N_effective (actual)  : {activations.shape[0]:,}")

## 8 — Normalization

Anthropic-style scalar normalization (TopK SAE paper):

$$x_{\text{norm}} = \frac{x}{\sqrt{\overline{x^2}} + \varepsilon} \cdot \sqrt{d_{\text{model}}}$$

This ensures $\mathbb{E}[\|x_{\text{norm}}\|^2] \approx d_{\text{model}}$.

In [ ]:
avg_sq_before = activations.pow(2).sum(dim=-1).mean().item()

rms          = activations.pow(2).mean(dim=-1, keepdim=True).sqrt() + EPS
activations  = activations / rms * math.sqrt(D_MODEL)

avg_sq_after = activations.pow(2).sum(dim=-1).mean().item()

print(f"avg ‖x‖²  before : {avg_sq_before:.2f}")
print(f"avg ‖x‖²  after  : {avg_sq_after:.2f}  (target ≈ {D_MODEL})")
print(f"dtype            : {activations.dtype}")
assert abs(avg_sq_after - D_MODEL) / D_MODEL < 0.05, \
    f"Normalization check failed: {avg_sq_after:.2f} vs expected {D_MODEL}"

## 9 — Save to disk

Saved as `data/dummy_activations.safetensors` — a memory-mapped, safe format
that Bilal can load directly with `safetensors.torch.load_file()`.

In [ ]:
out_path = Path(OUTPUT_PATH).resolve()
out_path.parent.mkdir(parents=True, exist_ok=True)

metadata = {
    "checkpoint" : str(CHECKPOINT),
    "d_model"    : str(D_MODEL),
    "spatial_h"  : str(H_FEAT),
    "spatial_w"  : str(W_FEAT),
    "n_images"   : str(N_images),
    "n_effective": str(activations.shape[0]),
    "pm_index"   : str(PM_INDEX),
    "image_size" : str(IMAGE_SIZE),
}

save_file({"activations": activations}, str(out_path), metadata=metadata)

size_mb = out_path.stat().st_size / 1e6
print(f"Saved  → {out_path}")
print(f"Size   : {size_mb:.1f} MB")
print(f"Shape  : {tuple(activations.shape)}")
print(f"\nMetadata:")
for k, v in metadata.items():
    print(f"  {k:<14} = {v}")

## 10 — Summary & Verification

In [ ]:
from safetensors.torch import load_file as st_load

# Reload and verify round-trip
loaded     = st_load(str(out_path))
acts_check = loaded["activations"]

print("=" * 55)
print("TopK SAE Activation Harvest — Summary")
print("=" * 55)
print(f"  Checkpoint   : {CHECKPOINT}")
print(f"  Dataset      : {len(dataset):,} synthetic images")
print(f"  Hook target  : PatchMerging PM-{PM_INDEX}  ({_boundary_names.get(PM_INDEX, '')}  boundary)")
print(f"  d_model      : {D_MODEL}")
print(f"  Spatial      : {H_FEAT}×{W_FEAT} = {N_SPATIAL} tokens/image")
print(f"  N_effective  : {activations.shape[0]:,}")
print(f"  Shape        : {tuple(acts_check.shape)}")
print(f"  dtype        : {acts_check.dtype}")
print(f"  File size    : {size_mb:.1f} MB")
print(f"  Output       : {out_path}")
print("=" * 55)
print()

# Sanity checks
assert acts_check.shape == activations.shape, "Shape mismatch after reload"
assert acts_check.dtype == torch.float32
assert not acts_check.isnan().any(), "NaNs in saved activations"
print("All checks passed.")
print(f"\n→ Bilal: {out_path} is ready for SAE training.")